# STIR-Net V1 — 26 Napari Visualization of Spatial-Proposal Masks

This notebook is a **visual debugging notebook** for the Notebook-24 spatial-proposal architecture.

It answers:

> What spatial proposals does STIR-Net create inside source component 9, and what 3-D masks do those proposal queries actually produce?

The notebook intentionally uses a **spatial-only forward**:

```text
raw / foreground / EDT / boundary / marker
        ↓
spatial encoder / decoder
        ↓
learned spatial proposals
        ↓
proposal queries
        ↓
query decoder
        ↓
native-resolution proposal masks
```

Temporal memory and CR1/CR2 are bypassed.

## Napari layers

The viewer contains:

- normalized raw 3-D volume
- current instance labels
- GT labels
- current source-9 component
- GT cells overlapping source 9
- proposal-score volume
- learned proposal anchors
- initial proposal-query anchors
- final predicted query centers
- combined predicted proposal masks
- optional one-layer-per-query masks
- foreground / EDT / marker / boundary inputs

By default the viewer crops around source 9 to keep the native mask rendering fast and the 3-D view readable.

> **Important:** this notebook loads the step-30 spatial checkpoint by default.  
> If you later save a better Notebook-24 checkpoint, set `CUSTOM_CHECKPOINT` below to that file.


In [ ]:
from pathlib import Path
import gc
import json
import math

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

try:
    import napari
except ImportError as exc:
    raise ImportError(
        "napari is not installed in this environment. "
        "Install it with: pip install 'napari[all]'"
    ) from exc

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.matcher import target_ids
from learned.stirnet.model.native_masks import compose_native_query_logits
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.model.types import StirNetOutput, TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.trainer import move_batch_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

# ---------------------------------------------------------------------
# Visualization options
# ---------------------------------------------------------------------

CROP_TO_SOURCE9 = True
CROP_MARGIN_DREF = 2.0

# Hard-mask threshold used only for the combined label visualization.
MASK_THRESHOLD = 0.50

# Show at most this many individual query-mask layers.
# All selected source-9 proposal queries are still used for the combined labels.
MAX_INDIVIDUAL_MASK_LAYERS = 12

# Set to a saved Notebook-24-style checkpoint if you have one.
# Leave as None to use the original step-30 spatial checkpoint.
CUSTOM_CHECKPOINT = None

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

DEFAULT_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

CHECKPOINT = (
    Path(CUSTOM_CHECKPOINT)
    if CUSTOM_CHECKPOINT is not None
    else DEFAULT_CHECKPOINT
)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 26 requires CUDA for the STIR-Net forward pass.")

if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")

device = torch.device("cuda")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", CHECKPOINT)
print("GPU        :", torch.cuda.get_device_name(0))


## 1. Load the same BlastoSPIM scene used in Notebooks 19–25


In [ ]:
batch_cpu, sample_info = build_real_batch(DATA_DIR)

b = move_batch_to_device(batch_cpu, device)
b["spatial_inputs"] = b["spatial_inputs"].to(dtype=AMP_DTYPE)
b["instance_labels"] = b["instance_labels"].to(dtype=torch.int32)

targets = batch_cpu["targets"]
target = targets[0]

spatial_inputs_native = (
    batch_cpu["spatial_inputs"][0]
    .detach()
    .cpu()
    .float()
    .numpy()
)

raw_native = spatial_inputs_native[0]
input_foreground_native = spatial_inputs_native[1]
edt_native = spatial_inputs_native[2]
input_boundary_native = spatial_inputs_native[3]
marker_native = spatial_inputs_native[4]

current_labels_native = (
    batch_cpu["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

gt_labels_native = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

spacing_native = (
    batch_cpu["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

dref_um = float(batch_cpu["dref_um"][0])

gt_ids = target_ids(target).detach().cpu().long()

source9_gt_ids = np.unique(
    gt_labels_native[current_labels_native == SOURCE_ID]
)
source9_gt_ids = source9_gt_ids[source9_gt_ids > 0].astype(int)

print(json.dumps(sample_info, indent=2, default=float))
print()
print("Source-9 GT IDs :", source9_gt_ids.tolist())
print("GT count        :", len(source9_gt_ids))
print("Volume shape    :", current_labels_native.shape)
print("Spacing z/y/x   :", spacing_native.tolist())
print("dref_um         :", dref_um)


## 2. Spatial-only STIR-Net forward

This uses the same strict spatial-only path as Notebook 24:

- no temporal memory
- no CR1
- no CR2
- spatial proposals enabled


In [ ]:
def make_empty_temporal(model, *, dtype):
    d_model = int(model.cfg.temporal.d_model)
    return TemporalState(
        tokens=torch.empty((0, d_model), device=device, dtype=dtype),
        ref_um=torch.empty((0, 3), device=device, dtype=torch.float32),
        ref_cellscale=torch.empty((0, 3), device=device, dtype=torch.float32),
        salience=torch.empty((0, 1), device=device, dtype=dtype),
        reliability=torch.empty((0, 1), device=device, dtype=dtype),
        status=torch.empty((0,), device=device, dtype=torch.long),
        edge_index=torch.empty((2, 0), device=device, dtype=torch.long),
        edge_attr=torch.empty((0, 22), device=device, dtype=torch.float32),
        batch_index=torch.empty((0,), device=device, dtype=torch.long),
    )


def load_model():
    cfg = _reduced_config()
    cfg.proposals.enabled = True
    cfg.proposals.query_mode = "spatial_proposals"

    model = StirNet(cfg).to(device)

    info = load_checkpoint(
        CHECKPOINT,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    return model, cfg, info


def forward_spatial_only_full(model):
    acq = model.acquisition(
        b["spacing_um"],
        b["dref_um"],
    )

    pyramid = model.encoder(
        b["spatial_inputs"],
        b["spacing_um"],
        acq,
        b.get("spatial_padding_mask"),
    )

    # Strictly spatial-only: bypass CR1/CR2.
    e3 = pyramid.features[3]

    e2 = model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    d1, d0, mask_features = model.decoder.decode_from_e2(
        e2,
        pyramid,
        acq,
    )

    dense = model.dense_heads(d0)

    proposal_state, proposal_score_logits = (
        model.spatial_proposal_generator(
            d0,
            e2,
            b["spatial_inputs"],
            dense,
            b["instance_labels"],
            b["spacing_um"],
            pyramid.spacings_um[2],
            b["dref_um"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b.get("spatial_padding_mask"),
        )
    )

    dense = dict(dense)
    dense["proposal_score_logits"] = proposal_score_logits

    temporal = make_empty_temporal(
        model,
        dtype=e2.dtype,
    )

    qstate = model.query_builder(
        e2,
        pyramid.spacings_um[2],
        b["instance_labels"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
        proposal_state=proposal_state,
        query_mode="spatial_proposals",
    )

    initial_references = qstate.references_cellscale.clone()

    qstate, decoder_outputs = model.query_decoder(
        qstate,
        [e3, e2, d1],
        [
            pyramid.spacings_um[3],
            pyramid.spacings_um[2],
            pyramid.spacings_um[1],
        ],
        b["instance_labels"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
    )

    final = decoder_outputs[-1]
    native_embeddings = model.native_mask_head(qstate.embeddings)

    return StirNetOutput(
        exist_logits=final["exist_logits"],
        centers_cellscale=final["centers_cellscale"],
        coarse_mask_logits=final["coarse_mask_logits"],
        coarse_spacing_um=final["coarse_spacing_um"],
        query_embeddings=qstate.embeddings,
        native_mask_embeddings=native_embeddings,
        query_types=qstate.query_types,
        query_padding_mask=qstate.padding_mask,
        source_instance_ids=qstate.source_instance_ids,
        query_initial_references_cellscale=initial_references,
        temporal_salience=qstate.temporal_salience,
        temporal_reliability=qstate.temporal_reliability,
        aux_outputs=decoder_outputs[:-1],
        dense_outputs=dense,
        mask_features=mask_features,
        spacing_um=b["spacing_um"],
        dref_um=b["dref_um"],
        instance_labels=b["instance_labels"],
        debug={
            "proposal_score_logits": proposal_score_logits,
        },
        proposals=proposal_state,
    )


model, cfg, load_info = load_model()
model.eval()

torch.cuda.reset_peak_memory_stats()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    outputs = forward_spatial_only_full(model)

print("Checkpoint step:", load_info.get("step"))
print(
    "Peak CUDA memory:",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB",
)


## 3. Select the source-9 proposal queries

These are the proposal queries whose provisional source association is the merged component `9`.


In [ ]:
valid = ~outputs.query_padding_mask[0].detach().cpu()
qtypes = outputs.query_types[0].detach().cpu()
sources = outputs.source_instance_ids[0].detach().cpu()

source9_query_mask = (
    valid
    & (qtypes == QUERY_SPATIAL_PROPOSAL)
    & (sources == SOURCE_ID)
)

source9_query_indices = torch.nonzero(
    source9_query_mask,
    as_tuple=False,
).flatten()

if source9_query_indices.numel() == 0:
    raise RuntimeError("No source-9 spatial proposal queries were produced.")

exist_probs = (
    outputs.exist_logits[0, source9_query_indices.to(device)]
    .detach()
    .float()
    .sigmoid()
    .cpu()
)

initial_refs = (
    outputs.query_initial_references_cellscale[0, source9_query_indices.to(device)]
    .detach()
    .float()
    .cpu()
)

final_refs = (
    outputs.centers_cellscale[0, source9_query_indices.to(device)]
    .detach()
    .float()
    .cpu()
)

query_table = pd.DataFrame(
    {
        "local_id": np.arange(len(source9_query_indices)),
        "query_index": source9_query_indices.numpy(),
        "exist_prob": exist_probs.numpy(),
        "initial_z_dref": initial_refs[:, 0].numpy(),
        "initial_y_dref": initial_refs[:, 1].numpy(),
        "initial_x_dref": initial_refs[:, 2].numpy(),
        "final_z_dref": final_refs[:, 0].numpy(),
        "final_y_dref": final_refs[:, 1].numpy(),
        "final_x_dref": final_refs[:, 2].numpy(),
    }
).sort_values("exist_prob", ascending=False)

display(query_table)

print("Source-9 proposal queries:", len(source9_query_indices))


## 4. Build a compact source-9 crop

Native proposal masks are rendered only inside this crop.  
For spatial-proposal queries the production semantics are radial support around each learned/refined anchor with zero source-mask prior.


In [ ]:
shape = np.asarray(current_labels_native.shape, dtype=np.int64)

source_voxels = np.argwhere(current_labels_native == SOURCE_ID)

if len(source_voxels) == 0:
    raise RuntimeError("Source component 9 is empty.")

if CROP_TO_SOURCE9:
    lo = source_voxels.min(axis=0)
    hi = source_voxels.max(axis=0) + 1

    margin_um = CROP_MARGIN_DREF * dref_um
    margin_vox = np.ceil(
        margin_um / spacing_native
    ).astype(np.int64)

    lo = np.maximum(0, lo - margin_vox)
    hi = np.minimum(shape, hi + margin_vox)
else:
    lo = np.zeros(3, dtype=np.int64)
    hi = shape.copy()

crop_slices = tuple(
    slice(int(a), int(b))
    for a, b in zip(lo, hi)
)

crop_shape = tuple((hi - lo).tolist())

raw_crop = raw_native[crop_slices]
current_crop = current_labels_native[crop_slices]
gt_crop = gt_labels_native[crop_slices]

source9_crop = (current_crop == SOURCE_ID).astype(np.uint8)
source9_gt_crop = np.isin(
    gt_crop,
    source9_gt_ids,
).astype(np.uint8)

proposal_score_crop = (
    outputs.dense_outputs["proposal_score_logits"][0, 0]
    .detach()
    .float()
    .cpu()
    .numpy()[crop_slices]
)

foreground_crop = input_foreground_native[crop_slices]
edt_crop = edt_native[crop_slices]
boundary_crop = input_boundary_native[crop_slices]
marker_crop = marker_native[crop_slices]

print("Crop start z/y/x :", lo.tolist())
print("Crop end   z/y/x :", hi.tolist())
print("Crop shape       :", crop_shape)


## 5. Render native-resolution proposal masks inside the crop

This is memory-conscious:

- the crop is rendered instead of the full `59×662×703` lattice;
- proposal queries are rendered one at a time;
- each probability volume is immediately moved to CPU.

The mask logits use the same proposal semantics as the model:

```text
learned mask embedding · native mask features
+
zero source prior
+
radial proposal support
```


In [ ]:
def crop_coordinates_um(lo, hi, spacing_um, full_shape):
    z = torch.arange(
        int(lo[0]), int(hi[0]),
        device=device,
        dtype=torch.float32,
    )
    y = torch.arange(
        int(lo[1]), int(hi[1]),
        device=device,
        dtype=torch.float32,
    )
    x = torch.arange(
        int(lo[2]), int(hi[2]),
        device=device,
        dtype=torch.float32,
    )

    zz, yy, xx = torch.meshgrid(
        z, y, x, indexing="ij"
    )

    coords_vox = torch.stack(
        [zz, yy, xx],
        dim=-1,
    ).reshape(-1, 3)

    spacing = torch.as_tensor(
        spacing_um,
        device=device,
        dtype=torch.float32,
    )

    extent = (
        torch.as_tensor(
            np.asarray(full_shape) - 1,
            device=device,
            dtype=torch.float32,
        )
        * spacing
    )

    return (
        coords_vox * spacing[None]
        - 0.5 * extent[None]
    )


coords_um = crop_coordinates_um(
    lo,
    hi,
    spacing_native,
    current_labels_native.shape,
)

crop_mask_features = (
    outputs.mask_features[
        0,
        :,
        crop_slices[0],
        crop_slices[1],
        crop_slices[2],
    ]
)

current_label_crop_t = (
    b["instance_labels"][
        0,
        crop_slices[0],
        crop_slices[1],
        crop_slices[2],
    ]
    .flatten()
)

source9_native_probabilities = []

for local_id, query_idx_cpu in enumerate(source9_query_indices):
    query_idx = int(query_idx_cpu)

    emb = (
        outputs.native_mask_embeddings[
            0, query_idx
        ]
        .float()
        .unsqueeze(0)
    )

    learned_logits = torch.einsum(
        "qc,cv->qv",
        emb,
        crop_mask_features.float().flatten(1),
    )

    selected_type = outputs.query_types[
        0, query_idx
    ].reshape(1)

    selected_source = outputs.source_instance_ids[
        0, query_idx
    ].reshape(1)

    ref_um = (
        outputs.centers_cellscale[
            0, query_idx
        ]
        .float()
        .reshape(1, 3)
        * b["dref_um"][0].float()
    )

    # Spatial-proposal queries do not use source-component support.
    source_support = torch.zeros(
        (1, coords_um.shape[0]),
        device=device,
        dtype=torch.bool,
    )

    logits, _, support = compose_native_query_logits(
        learned_logits,
        selected_type,
        selected_source,
        ref_um,
        current_label_crop_t,
        source_support,
        coords_um,
        b["dref_um"][0],
        support_radius_dref=1.5,
        temporal_sigma_dref=0.75,
        prior_inside_logit=1.5,
        prior_outside_logit=-1.5,
        background_logit=-20.0,
        proposal_support_radius_dref=(
            cfg.proposals.native_support_radius_dref
        ),
    )

    prob = (
        logits[0]
        .reshape(crop_shape)
        .sigmoid()
        .detach()
        .to("cpu", dtype=torch.float16)
        .numpy()
    )

    source9_native_probabilities.append(prob)

    del emb, learned_logits, logits, support
    torch.cuda.empty_cache()

source9_native_probabilities = np.stack(
    source9_native_probabilities,
    axis=0,
)

print(
    "Rendered native proposal masks:",
    source9_native_probabilities.shape,
)


## 6. Combine the predicted masks into a single label volume

Each voxel is assigned to the proposal query with the highest probability, but only when that maximum probability is at least `MASK_THRESHOLD`.

This is only for visualization; it is not the competition post-processing logic.


In [ ]:
best_local = source9_native_probabilities.argmax(axis=0)
best_prob = source9_native_probabilities.max(axis=0)

predicted_labels_crop = np.where(
    best_prob >= MASK_THRESHOLD,
    best_local + 1,
    0,
).astype(np.int32)

predicted_mask_count = int(
    np.unique(predicted_labels_crop).size
    - (1 if (predicted_labels_crop == 0).any() else 0)
)

print("Visible predicted mask IDs :", predicted_mask_count)
print("Mask threshold             :", MASK_THRESHOLD)
print(
    "Foreground voxels predicted:",
    int((predicted_labels_crop > 0).sum()),
)


## 7. Convert proposal/query coordinates to crop voxel coordinates


In [ ]:
def refs_cellscale_to_full_voxels(refs_cellscale):
    refs = torch.as_tensor(
        refs_cellscale,
        dtype=torch.float32,
    ).cpu().numpy()

    refs_um = refs * dref_um

    full_shape = np.asarray(
        current_labels_native.shape,
        dtype=np.float64,
    )

    patch_center_um = (
        0.5
        * (full_shape - 1)
        * spacing_native
    )

    return (
        refs_um
        + patch_center_um[None]
    ) / spacing_native[None]


initial_points_full = refs_cellscale_to_full_voxels(
    initial_refs
)
final_points_full = refs_cellscale_to_full_voxels(
    final_refs
)

initial_points_crop = initial_points_full - lo[None]
final_points_crop = final_points_full - lo[None]

proposal_state = outputs.proposals
proposal_valid = ~proposal_state.padding_mask[0].detach().cpu()
proposal_sources = proposal_state.source_instance_ids[0].detach().cpu()

proposal_source9 = (
    proposal_valid
    & (proposal_sources == SOURCE_ID)
)

proposal_refs = (
    proposal_state.references_cellscale[
        0
    ]
    .detach()
    .float()
    .cpu()[proposal_source9]
)

proposal_points_full = refs_cellscale_to_full_voxels(
    proposal_refs
)
proposal_points_crop = proposal_points_full - lo[None]

print("Proposal anchors :", len(proposal_points_crop))
print("Query anchors    :", len(initial_points_crop))


# 8. Open the Napari 3-D viewer

Recommended inspection:

1. Turn on **Current source 9** and **GT source-9 cells**.
2. Toggle **Predicted proposal masks**.
3. Toggle **Proposal anchors**, **Initial query anchors**, and **Final query centers**.
4. Inspect individual proposal-mask layers one at a time.
5. Switch between 2-D slices and Napari's 3-D view.

The individual mask layers are ordered by query existence probability.


In [ ]:
viewer = napari.Viewer(
    title="STIR-Net Notebook 26 — Spatial Proposal Masks",
    ndisplay=3
)

scale = tuple(spacing_native.tolist())

viewer.add_image(
    raw_crop,
    name="Raw normalized",
    scale=scale,
    blending="additive",
)

viewer.add_labels(
    current_crop,
    name="Current instance labels",
    scale=scale,
    visible=False,
)

viewer.add_labels(
    gt_crop,
    name="GT instance labels",
    scale=scale,
    visible=False,
)

viewer.add_labels(
    source9_crop,
    name="Current source 9",
    scale=scale,
    visible=True,
)

viewer.add_labels(
    source9_gt_crop,
    name="GT source-9 cells binary",
    scale=scale,
    visible=False,
)

viewer.add_labels(
    predicted_labels_crop,
    name="Predicted proposal masks",
    scale=scale,
    visible=True,
)

viewer.add_image(
    best_prob,
    name="Predicted max mask probability",
    scale=scale,
    colormap="magma",
    opacity=0.55,
    visible=False,
)

viewer.add_image(
    proposal_score_crop,
    name="Proposal score",
    scale=scale,
    colormap="viridis",
    opacity=0.6,
    visible=False,
)

viewer.add_image(
    edt_crop,
    name="Input EDT",
    scale=scale,
    colormap="turbo",
    visible=False,
)

viewer.add_image(
    marker_crop,
    name="Input marker heatmap",
    scale=scale,
    colormap="magma",
    visible=False,
)

viewer.add_image(
    boundary_crop,
    name="Input boundary",
    scale=scale,
    colormap="cyan",
    visible=False,
)

viewer.add_image(
    foreground_crop,
    name="Input foreground",
    scale=scale,
    colormap="gray",
    visible=False,
)

viewer.add_points(
    proposal_points_crop,
    name="Proposal anchors",
    scale=scale,
    size=2.5,
    symbol="disc",
    face_color="yellow",
)

viewer.add_points(
    initial_points_crop,
    name="Initial query anchors",
    scale=scale,
    size=3.0,
    symbol="ring",
    face_color="transparent",
    edge_color="lime",
)

viewer.add_points(
    final_points_crop,
    name="Final query centers",
    scale=scale,
    size=3.0,
    symbol="cross",
    face_color="red",
    edge_color="red",
)

# Add individual query masks in descending existence-probability order.
ranked_local_ids = (
    np.argsort(-exist_probs.numpy())
    [:MAX_INDIVIDUAL_MASK_LAYERS]
)

for rank, local_id in enumerate(ranked_local_ids):
    query_idx = int(source9_query_indices[int(local_id)])
    exist = float(exist_probs[int(local_id)])

    viewer.add_image(
        source9_native_probabilities[int(local_id)],
        name=(
            f"Q{query_idx:03d} mask prob "
            f"(exist={exist:.3f})"
        ),
        scale=scale,
        colormap="magma",
        opacity=0.7,
        visible=False,
    )

print("Napari viewer created.")
print(
    "If the window does not appear automatically in your IDE, "
    "run `napari.run()` in a new cell."
)


## 9. Optional: inspect the predicted masks numerically

This table reports simple crop-level visualization statistics.  
It is **not** the final support-aligned mask evaluation discussed after Notebook 24.


In [ ]:
rows = []

for local_id, query_idx in enumerate(source9_query_indices.tolist()):
    prob = source9_native_probabilities[local_id]
    hard = prob >= MASK_THRESHOLD

    rows.append(
        {
            "local_id": local_id,
            "query_index": int(query_idx),
            "exist_prob": float(exist_probs[local_id]),
            "mask_voxels": int(hard.sum()),
            "mean_prob_inside_hard": (
                float(prob[hard].mean())
                if hard.any()
                else np.nan
            ),
            "max_prob": float(prob.max()),
        }
    )

mask_table = (
    pd.DataFrame(rows)
    .sort_values("exist_prob", ascending=False)
    .reset_index(drop=True)
)

display(mask_table)


## What to look for

The key visual question is **not** whether the proposal queries have low cosine similarity.

Look for this instead:

### Good geometry, bad mask decoding

```text
proposal anchors:
●   ●   ●   ●   ●   ●   ●   ●   ●

but predicted masks:
██████████████████████████████████
```

That means the current bottleneck is downstream in anchor-aware query/mask decoding.

### Good geometry and separated masks

```text
proposal anchors:
●   ●   ●   ●   ●   ●   ●   ●   ●

predicted masks:
AA | BB | CC
DD | EE | FF
GG | HH | II
```

That means the Notebook-24 proposal architecture is already solving much more of the decomposition than the scalar mask metrics suggested.

### Bad proposal geometry

If several anchors cluster around the same real cell and other GT cells have none, the proposal mechanism is still the limiting factor.

For the current source-9 case, Notebook 24 already suggested the proposal geometry itself is strong, so the first two outcomes are the most informative.
